# 00 · Setup de ElectroCasa

Corre este notebook **una vez por target** antes del primer `databricks bundle deploy`:
con `catalog = electrocasa_dev` y después con `catalog = electrocasa_prod`.

Crea el catálogo, los schemas por capa, el volume de landing con una carpeta por fuente,
las tablas Bronze que carga COPY INTO, las funciones de máscara y los permisos de los tres grupos.

**Antes de correrlo:** crea los grupos `electrocasa_ingenieria`, `electrocasa_analistas` y
`electrocasa_auditoria` (Account console > User management > Groups, o Settings > Identity and access
del workspace si eres admin) y agrégate como miembro de `electrocasa_ingenieria`.

In [ ]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo del proyecto")
catalogo = dbutils.widgets.get("catalog")
print("Catálogo:", catalogo)

## 1. Catálogo, schemas y volume
Si tu metastore no tiene storage raíz, `CREATE CATALOG` pide `MANAGED LOCATION`: en ese caso agrega
`MANAGED LOCATION 'abfss://<contenedor>@<cuenta>.dfs.core.windows.net/electrocasa'` usando una
external location que ya tengas (la de la Sesión 01). Todas las tablas del proyecto son managed.

In [ ]:
sql = """
CREATE CATALOG IF NOT EXISTS ${electrocasa.catalog}
  COMMENT 'Proyecto integrador ElectroCasa: lakehouse de ventas, RR.HH., reseñas, devoluciones y despachos';
USE CATALOG ${electrocasa.catalog};
CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Ingesta raw, sin lógica de negocio';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Datos limpios, estandarizados y deduplicados';
CREATE SCHEMA IF NOT EXISTS gold   COMMENT 'Tablas que responden las preguntas de negocio';
CREATE VOLUME IF NOT EXISTS default.vol_landing COMMENT 'Landing de archivos por fuente';
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
for carpeta in ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]:
    dbutils.fs.mkdirs(f"/Volumes/{catalogo}/default/vol_landing/{carpeta}/")
display(dbutils.fs.ls(f"/Volumes/{catalogo}/default/vol_landing/"))

## 2. Tablas Bronze de COPY INTO
COPY INTO escribe en tablas Delta comunes (no Streaming Tables), así que se crean aquí, fuera del pipeline.
Todo es texto, como llega de la fuente, más las columnas técnicas de auditoría.

In [ ]:
sql = """
CREATE TABLE IF NOT EXISTS ${electrocasa.catalog}.bronze.catalogo (
  producto_id STRING, nombre_producto STRING, categoria STRING, marca STRING,
  precio_lista STRING, activo STRING,
  _archivo_origen STRING, _lote STRING, _fecha_ingesta TIMESTAMP
) COMMENT 'Snapshots completos del catálogo de productos (COPY INTO)';

CREATE TABLE IF NOT EXISTS ${electrocasa.catalog}.bronze.sucursales (
  sucursal_id STRING, nombre_sucursal STRING, distrito STRING, ciudad STRING,
  region STRING, fecha_apertura STRING,
  _archivo_origen STRING, _lote STRING, _fecha_ingesta TIMESTAMP
) COMMENT 'Tabla de referencia de sucursales (COPY INTO)';
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

## 3. Funciones de máscara
Las usa la cláusula `MASK` del schema de `silver.empleados_eventos`, `silver.empleados` y `gold.empleados_vigentes`.
Tienen que existir antes de desplegar el pipeline.
- `dni`: solo `electrocasa_ingenieria` ve el valor real; el resto ve `*****` + los últimos 3 dígitos.
- `salario`: solo `electrocasa_ingenieria` ve el valor; el resto ve `NULL`.

In [ ]:
sql = """
CREATE OR REPLACE FUNCTION ${electrocasa.catalog}.silver.mascara_dni(dni STRING)
  RETURNS STRING
  COMMENT 'Enmascara el DNI salvo para electrocasa_ingenieria'
  RETURN CASE
    WHEN is_account_group_member('electrocasa_ingenieria') THEN dni
    WHEN dni IS NULL THEN NULL
    ELSE concat('*****', right(dni, 3))
  END;

CREATE OR REPLACE FUNCTION ${electrocasa.catalog}.silver.mascara_salario(salario DECIMAL(10,2))
  RETURNS DECIMAL(10,2)
  COMMENT 'Oculta el salario salvo para electrocasa_ingenieria'
  RETURN CASE WHEN is_account_group_member('electrocasa_ingenieria') THEN salario ELSE NULL END;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

## 4. Permisos por grupo
| Grupo | Acceso |
|---|---|
| `electrocasa_ingenieria` | Lectura y escritura sobre bronze, silver y gold (y el volume de landing) |
| `electrocasa_analistas` | Solo lectura sobre gold |
| `electrocasa_auditoria` | Solo lectura sobre gold + evidencia de auditoría y lineage (ver 03_gobierno) |

In [ ]:
sql = """
-- Ingeniería: lectura y escritura en las tres capas
GRANT USE CATALOG ON CATALOG ${electrocasa.catalog} TO `electrocasa_ingenieria`;
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION, EXECUTE
  ON SCHEMA ${electrocasa.catalog}.bronze TO `electrocasa_ingenieria`;
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION, EXECUTE
  ON SCHEMA ${electrocasa.catalog}.silver TO `electrocasa_ingenieria`;
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION, EXECUTE
  ON SCHEMA ${electrocasa.catalog}.gold TO `electrocasa_ingenieria`;
GRANT USE SCHEMA ON SCHEMA ${electrocasa.catalog}.default TO `electrocasa_ingenieria`;
GRANT READ VOLUME, WRITE VOLUME ON VOLUME ${electrocasa.catalog}.default.vol_landing TO `electrocasa_ingenieria`;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Analistas: solo lectura sobre gold
GRANT USE CATALOG ON CATALOG ${electrocasa.catalog} TO `electrocasa_analistas`;
GRANT USE SCHEMA, SELECT ON SCHEMA ${electrocasa.catalog}.gold TO `electrocasa_analistas`;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Auditoría: solo lectura sobre gold (el acceso a system.access está en 03_gobierno)
GRANT USE CATALOG ON CATALOG ${electrocasa.catalog} TO `electrocasa_auditoria`;
GRANT USE SCHEMA, SELECT ON SCHEMA ${electrocasa.catalog}.gold TO `electrocasa_auditoria`;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
SHOW GRANTS ON CATALOG ${electrocasa.catalog};
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
SHOW GRANTS ON SCHEMA ${electrocasa.catalog}.gold;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

## 5. Subir el lote 1
Sube a su carpeta del volume **solo** los archivos del lote 1 de cada fuente, el snapshot de marzo del
catálogo y `sucursales.csv`. Luego corre el job (o deja que lo dispare el trigger por llegada de archivo).